In [ ]:
# Setup: import core analytics libs and list Kaggle input files.
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [1]:
# -*- coding: utf-8 -*-
"""01-quickstart-llamatelemetry-v0-1-0-e3 (Corrected for Kaggle)"""

# Base imports and typing helpers used across the notebook.

import os
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, Optional, List


In [4]:
# Check GPU availability and CUDA/VRAM details in the Kaggle runtime.

print("=" * 72)
print("KAGGLE GPU ENVIRONMENT CHECK")
print("=" * 72)

gpu_result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
gpu_lines = [line for line in gpu_result.stdout.splitlines() if line.startswith("GPU")]

print(f"Detected GPUs: {len(gpu_lines)}")
for line in gpu_lines:
    print(" ", line)

print("\nCUDA version:")
print(subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout)

print("\nVRAM summary:")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

if len(gpu_lines) < 2:
    print("\nWarning: This notebook expects Kaggle GPU T4 x2.")
else:
    print("\nReady: dual-GPU Kaggle environment detected.")


KAGGLE GPU ENVIRONMENT CHECK
Detected GPUs: 2
  GPU 0: Tesla T4 (UUID: GPU-6a635550-9409-f5a7-b3f1-b2962612ada6)
  GPU 1: Tesla T4 (UUID: GPU-8ebebad8-29fd-c7c7-df66-9a55899688ab)

CUDA version:
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


VRAM summary:
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


Ready: dual-GPU Kaggle environment detected.


In [2]:
# Install RAPIDS + Graphistry deps and verify imports.


!pip install -q --extra-index-url=https://pypi.nvidia.com "cugraph-cu12==25.10.0" 

!pip install -q pyarrow pandas numpy scipy huggingface_hub "graphistry[ai]" 



try:
    import cudf, cugraph
    print(f"✅ cuDF {cudf.__version__}")
    print(f"✅ cuGraph {cugraph.__version__}")
except ImportError as e:
    print(f"⚠️ RAPIDS: {e}")

try:
    import graphistry
    print(f"✅ Graphistry {graphistry.__version__}")
except ImportError as e:
    print(f"⚠️ Graphistry: {e}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.6 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 168.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 146.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 67.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 151.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 96.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.5 MB 90.9 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.9.0+cu126 requires nvidia-cublas-cu12==12.6.4.1; platform_system == "Linux", but you have nvidia-cublas-cu12 12.9.1.4 which is incompatible.
torch 2.9.0+cu126 requires nvidia-cuda-nvrtc

In [3]:
# Install llamatelemetry from GitHub and print the installed version.

!pip install -q --no-cache-dir --force-reinstall git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

import llamatelemetry
print("\nllamatelemetry version:", llamatelemetry.__version__)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 311.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 317.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 384.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 337.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 288.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 340.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 287.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 369.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/20


🎯 llamatelemetry v0.1.1 First-Time Setup - Kaggle 2× T4 Multi-GPU

🎮 GPU Detected: Tesla T4 (Compute 7.5)
  ✅ Tesla T4 detected - Perfect for llamatelemetry v0.1.1!
🌐 Platform: Kaggle

📦 Downloading Kaggle 2× T4 binaries (~961 MB)...
    Features: FlashAttention + Tensor Cores + Multi-GPU tensor-split

➡️  Attempt 1: HuggingFace (llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz)
📥 Downloading v0.1.1 from HuggingFace Hub...
   Repo: waqasm86/llamatelemetry-binaries
   File: v0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/1.40G [00:00<?, ?B/s]

v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/130 [00:00<?, ?B/s]

🔐 Verifying SHA256 checksum...
   ✅ Checksum verified
📦 Extracting llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz...
Found 21 files in archive
Extracted 21 files to /root/.cache/llamatelemetry/extract_0.1.1
✅ Extraction complete!
  Found bin/ and lib/ under /root/.cache/llamatelemetry/extract_0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2
  Copied 13 binaries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12
  Copied 2 libraries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/lib
✅ Binaries installed successfully!


llamatelemetry version: 0.1.1


In [4]:
# Detect CUDA, load the model, run a sample inference, and show throughput.

import llamatelemetry as lt
cuda_info = lt.detect_cuda()
print(f"CUDA available: {cuda_info['available']}")
for gpu in cuda_info["gpus"]:
    print(f"  {gpu['name']} - {gpu['memory']} MB")

engine = lt.InferenceEngine(enable_telemetry=False)

engine.load_model("gemma-3-1b-Q4_K_M", auto_start=True)

result = engine.infer("Explain GPU tensor cores in two sentences.")
print(result.text)
print(f"Tokens/sec: {result.tokens_per_sec:.1f}")


CUDA available: True
  Tesla T4 - 15360 MiB MB
  Tesla T4 - 15360 MiB MB
Loading model: gemma-3-1b-Q4_K_M

Model: gemma-3-1b-Q4_K_M
Description: Gemma 3 1B instruct (Q4_K_M) - Good for 1GB VRAM
Size: 700 MB (~0.7 GB)
Minimum VRAM: 0.5 GB
Source: https://huggingface.co/unsloth/gemma-3-1b-it-GGUF
Cache location: /usr/local/lib/python3.12/dist-packages/llamatelemetry/models/gemma-3-1b-it-Q4_K_M.gguf



Download this model? [Y/n]:  y



This may take a while (700 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


gemma-3-1b-it-Q4_K_M.gguf:   0%|          | 0.00/806M [00:00<?, ?B/s]

✓ Model downloaded: gemma-3-1b-it-Q4_K_M.gguf

Auto-configuring optimal settings...
✓ Auto-configured for 15.0 GB VRAM
  GPU Layers: 99
  Context Size: 4096
  Batch Size: 2048
  Micro-batch Size: 512

Starting llama-server...
GPU Check:
  Platform: kaggle
  GPU: Tesla T4
  Compute Capability: 7.5
  Status: ✓ Compatible
  Command: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server -m /usr/local/lib/python3.12/dist-packages/llamatelemetry/models/gemma-3-1b-it-Q4_K_M.gguf --host 127.0.0.1 --port 8090 -ngl 99 -c 4096 --parallel 1 -b 2048 -ub 512
Starting llama-server...
  Executable: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server
  Model: gemma-3-1b-it-Q4_K_M.gguf
  GPU Layers: 99
  Context Size: 4096
  Server URL: http://127.0.0.1:8090
Waiting for server to be ready...... ✓ Ready in 3.0s

✓ Model loaded and ready for inference
  Server: http://127.0.0.1:8090
  GPU Layers: 99
  Context Size: 4096


GPU tensor cores are s

In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.



In [ ]:
# Placeholder: add follow-up setup or experiments here.

